# 06b. Model Comparison - Automobile Loan Default Prediction

Compares every candidate model using the cross-validation results logged in `experiments/results/`. All numbers come from stratified 5-fold CV on the training split at a 0.5 threshold. The held-out test set is not used in this notebook.


## 1. Setup


In [1]:
import os
import sys

project_root = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.path.abspath(".")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

results_dir = "experiments/results" if os.path.exists("experiments/results") else "../experiments/results"


## 2. Collect Results

Results are read from three files: the shared log, the Logistic Regression file, and the XGBoost baseline-vs-tuned file. Duplicate rows for the same model keep the most recent entry. Each model has a protocol status: `verified` means its CV run is confirmed to match the shared protocol; `unverified` means the source still needs checking.


In [2]:
frames = [
    pd.read_csv(os.path.join(results_dir, "experiments.csv")),
    pd.read_csv(os.path.join(results_dir, "logistic_regression.csv")),
    pd.read_csv(os.path.join(results_dir, "xgboost_baseline_vs_tuned.csv")),
]
all_results = pd.concat(frames, ignore_index=True).drop_duplicates(subset=['model_name'], keep='last')

PROTOCOL_STATUS = {
    "random_forest": "verified",
    "random_forest_tuned": "verified",
    "gaussian_naive_bayes": "verified",
    "logistic_regression": "verified",
    "XGBoost Baseline": "verified",
    "XGBoost Tuned": "verified",
    "decision_tree": "unverified",
}
all_results["protocol_status"] = all_results["model_name"].map(PROTOCOL_STATUS).fillna("unknown")

print(all_results[['model_name', 'protocol_status', 'roc_auc_mean']].to_string(index=False))


          model_name protocol_status  roc_auc_mean
       decision_tree      unverified      0.708183
gaussian_naive_bayes        verified      0.696815
       random_forest        verified      0.774053
 random_forest_tuned        verified      0.769818
 logistic_regression        verified      0.733998
    XGBoost Baseline        verified      0.763093
       XGBoost Tuned        verified      0.770509


## 3. Comparison Table

Models are ranked by mean ROC-AUC, with PR-AUC as the tie-breaker. Recall, precision and F1 are at 0.5.


In [3]:
def mean_pm_std(mean, std):
    return f"{mean:.3f} +/- {std:.3f}"

table = pd.DataFrame({
    "model": all_results["model_name"],
    "ROC-AUC": [mean_pm_std(m, s) for m, s in zip(all_results["roc_auc_mean"], all_results["roc_auc_std"])],
    "PR-AUC": [mean_pm_std(m, s) for m, s in zip(all_results["pr_auc_mean"], all_results["pr_auc_std"])],
    "recall": all_results["recall_mean"].round(3),
    "precision": all_results["precision_mean"].round(3),
    "F1": all_results["f1_mean"].round(3),
    "threshold": all_results["threshold"].round(3),
    "protocol_status": all_results["protocol_status"],
})

order = all_results.sort_values(['roc_auc_mean', 'pr_auc_mean'], ascending=False).index
table = table.loc[order].reset_index(drop=True)
table.insert(0, 'rank', range(1, len(table) + 1))
table


,rank,model,ROC-AUC,PR-AUC,recall,precision,F1,threshold,protocol_status
0,1,random_forest,0.774 +/- 0.004,0.358 +/- 0.009,0.163,0.632,0.260,0.5,verified
1,2,XGBoost Tuned,0.771 +/- 0.004,0.307 +/- 0.009,0.045,0.717,0.085,0.5,verified
2,3,random_forest_tuned,0.770 +/- 0.004,0.339 +/- 0.009,0.285,0.381,0.326,0.5,verified
3,4,XGBoost Baseline,0.763 +/- 0.004,0.261 +/- 0.010,0.022,0.622,0.042,0.5,verified
4,5,logistic_regression,0.734 +/- 0.005,0.212 +/- 0.005,0.668,0.153,0.249,0.5,verified
5,6,decision_tree,0.708 +/- 0.004,0.181 +/- 0.004,0.607,0.152,0.242,0.5,unverified
6,7,gaussian_naive_bayes,0.697 +/- 0.004,0.176 +/- 0.009,0.479,0.170,0.250,0.5,verified


**Finding:** Random Forest (untuned) ranks first on ROC-AUC (0.774 +/- 0.004) and PR-AUC (0.358 +/- 0.009). XGBoost Tuned is close on ROC-AUC (0.771 +/- 0.004), a gap of 0.003, which is within one standard deviation, but it is well behind on PR-AUC (0.307). At 0.5, both top models have low recall (0.163 for Random Forest, 0.045 for XGBoost Tuned), so the operating threshold will matter for the final choice. Logistic Regression is clearly behind on ROC-AUC (0.734). Gaussian NB ranks last on ROC-AUC (0.697).

LightGBM is not in the table, because it has no cross-validated result yet. The Decision Tree row is flagged `unverified` until its source is confirmed.

**Decision (proposed):** carry Random Forest (untuned) and XGBoost Tuned forward as the leading candidates for the next step, plots and final selection. LightGBM and the Decision Tree stay pending until verified. The team should confirm this shortlist.
